In [0]:
from modules.utils.date_utils import get_month_start_n_months_ago

In [0]:
# Get the first day of the month six months ago
six_months_ago_start = get_month_start_n_months_ago(7)
next_month_start = get_month_start_n_months_ago(6)

In [0]:
# Load the cleansed yellow taxi trip data from the Silver layer and filter to only include trips with a pickup datetime later than the start the date from six months ago
df_trips = spark.read.table("nyctaxi.02_silver.yellow_trips_cleansed").filter(
    f"tpep_pickup_datetime >= '{six_months_ago_start}' AND tpep_pickup_datetime < '{next_month_start}'"
)

# Load taxi zone lookup data from the Silver layer
df_zones = spark.read.table("nyctaxi.02_silver.taxi_zone_lookup")

In [0]:
# Performs a LEFT JOIN with the zones table to enrich the trips data with pickup location details (borough and zone)
# Selects all relevant trip columns while aliasing the newly added pickup location attributes for clarity
df_join_1 = df_trips.join(
    df_zones,
    df_trips.pu_location_id == df_zones.location_id,
    "left"
).select(
    df_trips.vendor,
    df_trips.tpep_pickup_datetime,
    df_trips.tpep_dropoff_datetime,
    df_trips.trip_duration,
    df_trips.passenger_count,
    df_trips.trip_distance,
    df_trips.rate_type,
    df_zones.borough.alias("pu_borough"),
    df_zones.zone.alias("pu_zone"),
    df_trips.do_location_id, # for reading in the location id
    df_trips.payment_type,
    df_trips.fare_amount,
    df_trips.extra,
    df_trips.mta_tax,
    df_trips.tolls_amount,
    df_trips.improvement_surcharge,
    df_trips.total_amount,
    df_trips.congestion_surcharge,
    df_trips.airport_fee,
    df_trips.cbd_congestion_fee,
    df_trips.processed_timestamp
)

In [0]:
# Joins the dataset df_join_1 with df_zones to resolve and name the exact drop-off locations (borough and zone names)
# This translates the destination IDs into human-readable names to complete the trip route profile
df_join_2 = df_join_1.join(
    df_zones,
    df_join_1.do_location_id == df_zones.location_id,
    "left"
).select(
    df_join_1.vendor,
    df_join_1.tpep_pickup_datetime,
    df_join_1.tpep_dropoff_datetime,
    df_join_1.trip_duration,
    df_join_1.passenger_count,
    df_join_1.trip_distance,
    df_join_1.rate_type,
    df_join_1.pu_borough,
    df_zones.borough.alias("do_borough"),
    df_join_1.pu_zone,
    df_zones.zone.alias("do_zone"),
    df_join_1.payment_type,
    df_join_1.fare_amount,
    df_join_1.extra,
    df_join_1.mta_tax,
    df_join_1.tolls_amount,
    df_join_1.improvement_surcharge,
    df_join_1.total_amount,
    df_join_1.congestion_surcharge,
    df_join_1.airport_fee,
    df_join_1.cbd_congestion_fee,
    df_join_1.processed_timestamp
)

In [0]:
# Save the fully enriched taxi trip data into the Silver layer as a permanent Delta table
# This creates a consolidated table with complete route details (pickup and drop-off names) for analytics
df_join_2.write.mode("append").saveAsTable("nyctaxi.02_silver.yellow_trips_enriched")